# T1 — Adapter Bake-off: does Clifford structure beat LoRA at equal budget?

The pivot test. Three PEFT adapters on a **frozen** SmolLM2-360M, same GSM8K data,
same schedule, same `q_proj`/`v_proj` targets — only the adapter math differs:

| adapter | math | trainable |
|---|---|---|
| **LoRA** | low-rank delta `B(A(x))` | ~1.64M (0.45%) |
| **Geo** | ReZero geometric-product residual `α·proj(mv ⊗ w)` | ~1.64M (matched) |
| **Rotor** | orthogonal rotor sandwich `R·base(x)·R̃` | ~15k (frugal) |

`Geo` is the direct test of the one GDR ember that survived the gated re-test (the
geometric-product cross-term). Both Clifford adapters are **exact identities at init**
(α=0 / bivector=0) — the same ReZero discipline that made the gated test fair.

**Metric:** held-out CE on GSM8K answer tokens (dense, low-variance — primary) +
test exact-match accuracy (capability check). **Read:** if Geo/Rotor don't beat LoRA
at matched budget, Clifford has no edge even as an adapter — pivot is pure capability.
Free-T4 runtime ~1–1.5 h for all three. Set GPU runtime.


In [ ]:
# Colab ships torch/transformers/datasets as a consistent set. Do NOT upgrade numpy
# (it breaks the prebuilt scipy/sklearn ABI). Install datasets only if missing.
import importlib, subprocess, sys
for pkg in ("transformers", "datasets"):
    if importlib.util.find_spec(pkg) is None:
        subprocess.run([sys.executable, "-m", "pip", "-q", "install", pkg], check=True)
import torch, transformers, datasets
print("torch", torch.__version__, "| transformers", transformers.__version__,
      "| datasets", datasets.__version__, "| cuda", torch.cuda.is_available())


In [ ]:
# --- knobs ---
BASE_MODEL     = "HuggingFaceTB/SmolLM2-360M"
TARGETS        = ("q_proj", "v_proj")   # same insertion points for all three adapters
LORA_RANK      = 16                      # LoRA r ; geo n_mv chosen to match its param count
GEO_NMV        = 2
ADAPTERS       = {"lora": dict(rank=LORA_RANK), "geo": dict(n_mv=GEO_NMV), "rotor": dict()}

TRAIN_EXAMPLES = 4000     # GSM8K train subset to tokenize
TRAIN_STEPS    = 400
BATCH          = 8
SEQ            = 320      # GSM8K Q+A fits comfortably
LR             = 2e-4
WARMUP         = 30
EVAL_CE_N      = 300      # held-out test examples for CE (primary metric)
EVAL_EM_N      = 150      # test examples for generate+exact-match (slower)
SEED           = 1234
OUT_DIR        = "t1_out"  # results.json + small adapter weights saved here per adapter
GRAD_CKPT      = False     # set True for low-VRAM GPUs (frozen 360M fits 4GB with this on)
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
assert DEVICE == "cuda", "set Colab runtime to GPU (T4)"

# IMPORTANT: SmolLM2 in fp16 overflows to NaN logits on some GSM8K sequences.
# Use bf16 on Ampere+ (cc>=8: A100/L4) for speed, fp32 on older cards (T4, cc 7.5)
# for correctness. Never fp16.
_CC = torch.cuda.get_device_capability()[0]
DTYPE = torch.bfloat16 if _CC >= 8 else torch.float32
print(f"GPU {torch.cuda.get_device_name()} (cc {_CC}.x) -> dtype {DTYPE}")


### (optional) Persist results to Google Drive

Colab's local disk dies with the session. Run this to save `results.json` + the small
adapter weights to Drive so a disconnect mid-run loses nothing (the bug that ate the
earlier A100 run). Skip it and `OUT_DIR` stays on local disk.


In [ ]:
import os
# Uncomment to persist across disconnects:
# from google.colab import drive; drive.mount('/content/drive')
# OUT_DIR = '/content/drive/MyDrive/hagi_t1_out'
os.makedirs(OUT_DIR, exist_ok=True)
print('saving results + adapter weights to:', os.path.abspath(OUT_DIR))


In [ ]:
from __future__ import annotations
# ====================================================================
# Embedded verbatim from prototype/model/{clifford,clifford_adapters}.py
# (covered by prototype/tests/test_clifford_adapters.py). Standalone here
# so the notebook needs no repo clone.
# ====================================================================
"""Clifford algebra Cl(3,0,0) geometric product.

Cl(3,0,0): three orthonormal basis vectors e1, e2, e3, each squaring to +1.
8 basis blades indexed by 3-bit bitmask (bit i set => e_{i+1} present):

    0b000 = 1            (grade 0, scalar)
    0b001 = e1           (grade 1)
    0b010 = e2           (grade 1)
    0b100 = e3           (grade 1)
    0b011 = e1 e2        (grade 2, bivector)
    0b101 = e1 e3        (grade 2, bivector)
    0b110 = e2 e3        (grade 2, bivector)
    0b111 = e1 e2 e3     (grade 3, trivector / pseudoscalar)

The geometric product of two basis blades a, b (bitmasks):
    result_blade = a XOR b
    sign         = (-1)^(reordering transpositions)   [metric is all +1]

This module is the foundation of Grade-Decomposed Recurrence. It is pure,
deterministic, and verifiable — the Cayley table is checked against the Lean4
spec (`formalization/HAGI/HDIM.lean`).
"""


import torch

BLADE_COUNT = 8
DIM = 3

# Grade (popcount) of each blade index.
GRADE = [bin(i).count("1") for i in range(BLADE_COUNT)]  # [0,1,1,2,1,2,2,3]


def _reordering_sign(a: int, b: int) -> int:
    """Sign from reordering the product of two basis blades into canonical order.

    Counts transpositions needed to sort the concatenated basis vectors.
    Metric is Euclidean (+1) so shared indices contribute no extra sign.
    """
    a >>= 1
    swaps = 0
    while a:
        swaps += bin(a & b).count("1")
        a >>= 1
    return -1 if (swaps & 1) else 1


def build_product_table() -> tuple[torch.Tensor, torch.Tensor]:
    """Build the Cl(3,0,0) Cayley table.

    Returns:
        out_index: [8, 8] long tensor, out_index[a, b] = resulting blade index.
        sign:      [8, 8] float tensor, sign[a, b] = +1 or -1.
    """
    out_index = torch.zeros(BLADE_COUNT, BLADE_COUNT, dtype=torch.long)
    sign = torch.zeros(BLADE_COUNT, BLADE_COUNT, dtype=torch.float32)
    for a in range(BLADE_COUNT):
        for b in range(BLADE_COUNT):
            out_index[a, b] = a ^ b
            sign[a, b] = float(_reordering_sign(a, b))
    return out_index, sign


# Precomputed tables (module-level constants).
_OUT_INDEX, _SIGN = build_product_table()


def build_structure_constants() -> torch.Tensor:
    """Dense [8, 8, 8] structure-constant tensor C[a, b, c] for the Cayley table.

    C[a, b, c] = sign(a, b) if (a XOR b) == c else 0. Then the geometric product
    is one contraction: out[..., c] = sum_{a,b} x[..., a] * C[a, b, c] * y[..., b].
    """
    out_index, sign = build_product_table()
    c = torch.zeros(BLADE_COUNT, BLADE_COUNT, BLADE_COUNT, dtype=torch.float32)
    for a in range(BLADE_COUNT):
        for b in range(BLADE_COUNT):
            c[a, b, int(out_index[a, b])] = sign[a, b]
    return c


_STRUCT = build_structure_constants()


def geometric_product(x: torch.Tensor, y: torch.Tensor) -> torch.Tensor:
    """Geometric product of two batched multivectors.

    Args:
        x: [..., 8] multivector coefficients.
        y: [..., 8] multivector coefficients.

    Returns:
        [..., 8] product coefficients.

    Single fused einsum over the precomputed structure constants — vectorized,
    fp32-accumulated, and `torch.compile`-friendly (no Python blade loop and no
    `int(tensor)` host sync, which forced a graph break in the looped GDR core).
    """
    assert x.shape[-1] == BLADE_COUNT, f"expected last dim {BLADE_COUNT}, got {x.shape[-1]}"
    assert y.shape[-1] == BLADE_COUNT, f"expected last dim {BLADE_COUNT}, got {y.shape[-1]}"

    c = _STRUCT.to(x.device)  # [8,8,8] constant; dynamo constant-folds the device move
    out = torch.einsum("...a,abc,...b->...c", x.float(), c, y.float())
    return out.to(x.dtype)


def grade_projection(mv: torch.Tensor, grade: int) -> torch.Tensor:
    """Zero out all blades not of the given grade. Returns [..., 8]."""
    mask = torch.tensor(
        [1.0 if GRADE[i] == grade else 0.0 for i in range(BLADE_COUNT)],
        dtype=mv.dtype,
        device=mv.device,
    )
    return mv * mask


def reverse(mv: torch.Tensor) -> torch.Tensor:
    """Clifford reverse: sign (-1)^(k(k-1)/2) per grade k. Returns [..., 8]."""
    signs = torch.tensor(
        [(-1.0) ** (GRADE[i] * (GRADE[i] - 1) // 2) for i in range(BLADE_COUNT)],
        dtype=mv.dtype,
        device=mv.device,
    )
    return mv * signs


"""Parameter-efficient adapters for the T1 bake-off: does Clifford structure beat
a low-rank adapter at equal budget on a *frozen* pretrained base?

Three residual/multiplicative adapters, all injected at the same `nn.Linear`
targets so the only thing that differs is the adapter's internal math:

  - LoRAAdapter        : y = base(x) + scale * B(A(x))           (low-rank delta)
  - GeoProductAdapter  : y = base(x) + alpha * proj_out(mv ⊗ w)  (ReZero geo residual)
  - RotorAdapter       : y = R · base(x) · R̃                     (orthogonal rotor sandwich)

GeoProductAdapter is the direct test of the one GDR ember that survived the gated
re-test (the geometric-product cross-term — the only gate that moved off zero).
Both Clifford adapters start as EXACT identities (alpha=0 / bivector=0), the same
ReZero discipline that made the gated GDR test fair: the model must opt INTO the
geometric machinery.

Fairness is by reported trainable-param count (see `count_trainable`) + identical
data/schedule/targets, not by forcing identical algebra.
"""


import math

import torch
from torch import nn


# Even-grade (rotor) blade indices in the Cl(3,0,0) layout: scalar + the 3 bivectors.
_BIVECTOR_BLADES = (3, 5, 6)  # e1e2, e1e3, e2e3


def _broadcast_mv(weight: torch.Tensor, like: torch.Tensor) -> torch.Tensor:
    """Broadcast a per-channel multivector [..., n_mv, 8] against batched `like`."""
    return torch.broadcast_to(weight, like.shape)


class LoRAAdapter(nn.Module):
    """Standard low-rank delta on the input: delta = scale * B(A(x)), B init 0."""

    def __init__(self, in_features: int, out_features: int, rank: int = 16, alpha: float = 16.0):
        super().__init__()
        self.rank = rank
        self.A = nn.Linear(in_features, rank, bias=False)
        self.B = nn.Linear(rank, out_features, bias=False)
        nn.init.kaiming_uniform_(self.A.weight, a=math.sqrt(5))
        nn.init.zeros_(self.B.weight)  # delta == 0 at init
        self.scale = alpha / rank

    def forward(self, x: torch.Tensor, base_out: torch.Tensor) -> torch.Tensor:
        # Adapter params stay fp32 for stable training; base may be fp16/bf16.
        dt = self.A.weight.dtype
        delta = self.B(self.A(x.to(dt))) * self.scale
        return base_out + delta.to(base_out.dtype)


class GeoProductAdapter(nn.Module):
    """ReZero geometric-product residual. Projects x into n_mv multivectors, takes the
    geometric product with a learned per-channel multivector, projects back. alpha init
    0 -> exact identity at init (the gated-GDR discipline, now as an adapter)."""

    def __init__(self, in_features: int, out_features: int, n_mv: int = 2):
        super().__init__()
        self.n_mv = n_mv
        self.proj_in = nn.Linear(in_features, n_mv * BLADE_COUNT, bias=False)
        self.proj_out = nn.Linear(n_mv * BLADE_COUNT, out_features, bias=False)
        # Learned multivector the input is geometrically multiplied by (small init).
        self.weight_mv = nn.Parameter(torch.randn(n_mv, BLADE_COUNT) * 0.02)
        self.alpha = nn.Parameter(torch.zeros(1))  # ReZero gate
        nn.init.kaiming_uniform_(self.proj_in.weight, a=math.sqrt(5))
        nn.init.kaiming_uniform_(self.proj_out.weight, a=math.sqrt(5))

    def forward(self, x: torch.Tensor, base_out: torch.Tensor) -> torch.Tensor:
        dt = self.proj_in.weight.dtype  # adapter math in fp32, base may be fp16/bf16
        x = x.to(dt)
        *lead, _ = x.shape
        mv = self.proj_in(x).reshape(*lead, self.n_mv, BLADE_COUNT)
        w = _broadcast_mv(self.weight_mv, mv)
        prod = geometric_product(mv, w)  # [..., n_mv, 8]
        delta = self.alpha * self.proj_out(prod.reshape(*lead, self.n_mv * BLADE_COUNT))
        return base_out + delta.to(base_out.dtype)


class RotorAdapter(nn.Module):
    """Orthogonal fine-tuning via Clifford rotors. Reshapes base_out into n_mv 8-blade
    multivectors and applies the sandwich R v R̃ with a learned rotor R = exp(-B/2) per
    channel (B a learned bivector). bivector init 0 -> R = 1 -> identity at init.
    Param-frugal by design (3 angles/channel); raise coverage to match LoRA's budget."""

    def __init__(self, out_features: int, **_ignored):
        super().__init__()
        assert out_features % BLADE_COUNT == 0, "rotor target out_features must be divisible by 8"
        self.n_mv = out_features // BLADE_COUNT
        # One bivector (3 angles) per multivector channel; init 0 == identity rotor.
        self.bivector = nn.Parameter(torch.zeros(self.n_mv, len(_BIVECTOR_BLADES)))

    def _rotor(self) -> torch.Tensor:
        """Build R = exp(-B/2) as a multivector [n_mv, 8] from the learned bivectors."""
        biv = self.bivector
        phi = biv.norm(dim=-1, keepdim=True)            # [n_mv, 1]
        half = phi * 0.5
        # sin(half)/phi, with the well-defined limit 0.5 as phi -> 0.
        sinc = torch.where(phi > 1e-8, torch.sin(half) / phi.clamp_min(1e-8),
                           torch.full_like(phi, 0.5))
        R = torch.zeros(self.n_mv, BLADE_COUNT, device=biv.device, dtype=biv.dtype)
        R[:, 0] = torch.cos(half).squeeze(-1)
        for k, blade in enumerate(_BIVECTOR_BLADES):   # R = cos(half) - sinc * B
            R[:, blade] = -(sinc.squeeze(-1) * biv[:, k])
        return R

    def forward(self, x: torch.Tensor, base_out: torch.Tensor) -> torch.Tensor:
        dt = self.bivector.dtype  # rotor math in fp32, base may be fp16/bf16
        *lead, last = base_out.shape
        v = base_out.to(dt).reshape(*lead, self.n_mv, BLADE_COUNT)
        R = self._rotor()                               # [n_mv, 8]
        Rb = _broadcast_mv(R, v)
        Rt = _broadcast_mv(reverse(R), v)
        rotated = geometric_product(geometric_product(Rb, v), Rt)
        return rotated.reshape(*lead, last).to(base_out.dtype)


class WrappedLinear(nn.Module):
    """Frozen base Linear with an adapter applied to its output."""

    def __init__(self, base: nn.Linear, adapter: nn.Module):
        super().__init__()
        self.base = base
        self.adapter = adapter
        for p in self.base.parameters():
            p.requires_grad_(False)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.adapter(x, self.base(x))


_ADAPTERS = {"lora": LoRAAdapter, "geo": GeoProductAdapter, "rotor": RotorAdapter}


def _make_adapter(kind: str, lin: nn.Linear, **knobs) -> nn.Module:
    in_f, out_f = lin.in_features, lin.out_features
    if kind == "lora":
        return LoRAAdapter(in_f, out_f, rank=knobs.get("rank", 16), alpha=knobs.get("alpha", 16.0))
    if kind == "geo":
        return GeoProductAdapter(in_f, out_f, n_mv=knobs.get("n_mv", 2))
    if kind == "rotor":
        return RotorAdapter(out_f)
    raise ValueError(f"unknown adapter kind: {kind}")


def _set_submodule(root: nn.Module, dotted: str, value: nn.Module) -> None:
    parent = root
    *path, last = dotted.split(".")
    for p in path:
        parent = getattr(parent, p)
    setattr(parent, last, value)


def inject_adapters(model: nn.Module, kind: str, targets=("q_proj", "v_proj"),
                    freeze_base: bool = True, **knobs) -> dict:
    """Wrap every `nn.Linear` whose name ends in a target suffix with `kind` adapter.

    Returns a summary dict (n_wrapped, trainable params, skipped). Base is frozen.
    """
    if freeze_base:
        for p in model.parameters():
            p.requires_grad_(False)

    to_wrap, skipped = [], []
    for name, mod in model.named_modules():
        if isinstance(mod, nn.Linear) and any(name.endswith(t) for t in targets):
            if kind == "rotor" and mod.out_features % BLADE_COUNT != 0:
                skipped.append(name)
                continue
            to_wrap.append((name, mod))

    for name, lin in to_wrap:
        _set_submodule(model, name, WrappedLinear(lin, _make_adapter(kind, lin, **knobs)))

    return {
        "kind": kind,
        "n_wrapped": len(to_wrap),
        "skipped": skipped,
        "trainable": count_trainable(model),
    }


def count_trainable(model: nn.Module) -> int:
    return sum(p.numel() for p in model.parameters() if p.requires_grad)


In [ ]:
# GSM8K -> prompt-masked SFT tensors (train on the answer tokens only).
import re, random
from transformers import AutoTokenizer
from datasets import load_dataset

tok = AutoTokenizer.from_pretrained(BASE_MODEL)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token

gsm = load_dataset("openai/gsm8k", "main")
train_raw, test_raw = gsm["train"], gsm["test"]
PROMPT = "Question: {q}\nAnswer:"

def build_example(q, a):
    p = tok(PROMPT.format(q=q), add_special_tokens=True).input_ids
    ans = tok(" " + a + tok.eos_token, add_special_tokens=False).input_ids
    ids = (p + ans)[:SEQ]
    labels = ([-100] * len(p) + ans)[:SEQ]
    return ids, labels

random.seed(SEED)
n_train = min(TRAIN_EXAMPLES, len(train_raw))
train_items = [build_example(e["question"], e["answer"]) for e in train_raw.select(range(n_train))]
val_items   = [build_example(e["question"], e["answer"]) for e in test_raw.select(range(EVAL_CE_N))]
print(f"train items {len(train_items)} | val(CE) items {len(val_items)} | "
      f"median len {sorted(len(i[0]) for i in train_items)[len(train_items)//2]}")

def collate(items):
    m = max(len(i[0]) for i in items)
    ids = torch.full((len(items), m), tok.pad_token_id, dtype=torch.long)
    lab = torch.full((len(items), m), -100, dtype=torch.long)
    for j, (i, l) in enumerate(items):
        ids[j, :len(i)] = torch.tensor(i)
        lab[j, :len(l)] = torch.tensor(l)
    return ids, lab

def extract_final(t):
    m = re.search(r"####\s*(-?[\d,]+)", t)
    if m:
        return m.group(1).replace(",", "")
    nums = re.findall(r"-?\d[\d,]*", t)
    return nums[-1].replace(",", "") if nums else None


In [ ]:
# Train one adapter, then score held-out CE (answer tokens) + test exact-match.
import math, time
import torch.nn.functional as F
from transformers import AutoModelForCausalLM

def fresh_base():
    torch.manual_seed(SEED)
    m = AutoModelForCausalLM.from_pretrained(BASE_MODEL, dtype=DTYPE).to(DEVICE)
    if GRAD_CKPT:
        m.gradient_checkpointing_enable()
        m.enable_input_require_grads()   # base frozen + grad ckpt needs this
        m.config.use_cache = False
    return m

@torch.no_grad()
def eval_ce(model):
    model.eval(); tot = 0.0; ntok = 0
    for k in range(0, len(val_items), BATCH):
        ids, lab = collate(val_items[k:k + BATCH])
        ids, lab = ids.to(DEVICE), lab.to(DEVICE)
        logits = model(input_ids=ids).logits[:, :-1].float()
        tgt = lab[:, 1:]
        l = F.cross_entropy(logits.reshape(-1, logits.size(-1)), tgt.reshape(-1),
                            ignore_index=-100, reduction="sum")
        tot += l.item(); ntok += (tgt != -100).sum().item()
    return tot / max(ntok, 1)

@torch.no_grad()
def eval_em(model, n):
    model.eval(); correct = 0
    for e in test_raw.select(range(n)):
        enc = tok(PROMPT.format(q=e["question"]), return_tensors="pt").to(DEVICE)
        gen = model.generate(**enc, max_new_tokens=256, do_sample=False,
                             pad_token_id=tok.pad_token_id)
        comp = tok.decode(gen[0][enc.input_ids.shape[1]:], skip_special_tokens=True)
        correct += (extract_final(comp) == extract_final(e["answer"]))
    return correct / n

def run_adapter(kind, **knobs):
    model = fresh_base()
    info = inject_adapters(model, kind, targets=TARGETS, **knobs)
    model.to(DEVICE); model.train()
    params = [p for p in model.parameters() if p.requires_grad]
    opt = torch.optim.AdamW(params, lr=LR, weight_decay=0.0, betas=(0.9, 0.95))
    def lr_at(s):
        if s < WARMUP:
            return s / max(WARMUP, 1)
        p = (s - WARMUP) / max(TRAIN_STEPS - WARMUP, 1)
        return 0.5 * (1 + math.cos(math.pi * p))
    order = list(range(len(train_items))); random.shuffle(order)
    step = 0; ptr = 0; t0 = time.time(); last = 0.0
    while step < TRAIN_STEPS:
        if ptr + BATCH > len(order):
            random.shuffle(order); ptr = 0
        idxs = order[ptr:ptr + BATCH]; ptr += BATCH
        ids, lab = collate([train_items[j] for j in idxs])
        ids, lab = ids.to(DEVICE), lab.to(DEVICE)
        loss = model(input_ids=ids, labels=lab).loss
        for g in opt.param_groups:
            g["lr"] = LR * lr_at(step)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(params, 1.0)
        opt.step(); step += 1; last = loss.item()
        if step % 100 == 0 or step == 1:
            print(f"  [{kind}] step {step:4d}/{TRAIN_STEPS} loss {last:.3f} "
                  f"({(time.time()-t0)/step:.2f}s/step)")
    if GRAD_CKPT:
        model.config.use_cache = True  # generation wants the cache back
    ce = eval_ce(model)
    em = eval_em(model, EVAL_EM_N)
    # Persist the small adapter weights (trainable params only) so a disconnect
    # after this adapter still leaves something on disk.
    import os
    sd = {n: p.detach().cpu() for n, p in model.named_parameters() if p.requires_grad}
    torch.save(sd, os.path.join(OUT_DIR, f"{kind}_adapter.pt"))
    res = dict(kind=kind, trainable=info["trainable"], n_wrapped=info["n_wrapped"],
               train_loss=round(last, 4), held_out_ce=round(ce, 4),
               ppl=round(math.exp(ce), 3), test_em=round(em, 4))
    del model
    torch.cuda.empty_cache()
    print(f"  [{kind}] DONE  CE {ce:.4f} | ppl {math.exp(ce):.2f} | EM {em:.3f}")
    return res


In [ ]:
# Run the bake-off. Same data/schedule/targets; only the adapter math differs.
# results.json is rewritten after EACH adapter, so a disconnect keeps finished ones.
import json as _json, os
results = []
for kind, knobs in ADAPTERS.items():
    print(f"=== {kind} ===")
    results.append(run_adapter(kind, **knobs))
    _json.dump(results, open(os.path.join(OUT_DIR, "results.json"), "w"), indent=2)

print("\n================ T1 RESULTS ================")
hdr = f"{'adapter':<8}{'trainable':>12}{'held-out CE':>14}{'ppl':>10}{'test EM':>10}"
print(hdr); print("-" * len(hdr))
for r in sorted(results, key=lambda r: r["held_out_ce"]):
    print(f"{r['kind']:<8}{r['trainable']:>12,}{r['held_out_ce']:>14.4f}"
          f"{r['ppl']:>10.2f}{r['test_em']:>10.3f}")

lora = next(r for r in results if r["kind"] == "lora")
print("\nvs LoRA (held-out CE; negative = Clifford adapter wins):")
for r in results:
    if r["kind"] != "lora":
        d = r["held_out_ce"] - lora["held_out_ce"]
        print(f"  {r['kind']:<6} ΔCE = {d:+.4f}   ΔEM = {r['test_em']-lora['test_em']:+.3f}"
              f"   ({'WINS' if d < 0 else 'loses'} vs LoRA)")
import json as _json
print("\nJSON " + _json.dumps(results))


## Read

- **Primary = held-out CE on answer tokens** (dense, low-variance). Test EM is the
  capability check but noisy at 360M + light SFT — trust CE for the ranking, EM for
  the headline.
- **Geo vs LoRA at matched 1.64M params** is the clean head-to-head: does the
  geometric product add anything a low-rank delta can't? `Geo ΔCE < 0` ⇒ Clifford
  structure has real value as an adapter → scale to SmolLM2-1.7B (T2) and port the
  fuller grade/loop variants. `Geo ΔCE ≥ 0` ⇒ the ember was noise; pivot is pure
  capability (plain LoRA), Clifford retired for good.
- **Rotor** is a separate budget class (~15k params, 100× leaner). If it rivals LoRA
  at 100× fewer params, that is a genuine efficiency finding worth its own note; if it
  trails, part of that is the budget gap — not a clean refutation.

Bump `TRAIN_STEPS` / `EVAL_EM_N` for a tighter read; drop them if T4 time is short.
